# Stage 0 — Environment Check

**Purpose:** Verify that all required packages are installed and importable, confirm the dataset loads correctly, and validate basic data integrity before starting any analysis.

**Why this notebook exists:** Running the full pipeline requires ~12 Python packages working together. It's better to catch import errors, version mismatches, or missing data files here — in a 30-second check — than to discover them halfway through an EDA notebook.

**What this notebook checks:**
1. All required packages import successfully
2. Package versions meet minimum requirements
3. `cs-training.csv` loads from `data/raw/`
4. Dataset shape matches expectations (~150K rows, 12 columns including the unnamed index)
5. Column names and data types are as expected
6. Null counts per column
7. Target variable distribution and default rate

**If everything passes:** You're ready to proceed to `01_eda.ipynb`.

---
## 0.1 — Package Imports

Import every package that the pipeline will use. If any of these fail, run `pip install -r requirements.txt` from the project root.

In [ ]:
import sys
import importlib

# ── Core data manipulation ──
import pandas as pd
import numpy as np

# ── Statistical modelling ──
import statsmodels
import sklearn

# ── Gradient boosting ──
import xgboost

# ── Credit-risk-specific binning ──
import optbinning

# ── Explainability ──
import shap

# ── Visualisation ──
import matplotlib
import seaborn as sns

# ── Excel export ──
import openpyxl

print("All packages imported successfully.")

---
## 0.2 — Version Check

Print the version of every key package to confirm we meet the minimum requirements specified in `requirements.txt`. This also helps with reproducibility — if someone else runs this project, they can compare versions.

In [ ]:
packages = {
    "Python":       sys.version.split()[0],
    "pandas":       pd.__version__,
    "numpy":        np.__version__,
    "statsmodels":  statsmodels.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost":      xgboost.__version__,
    "optbinning":   optbinning.__version__,
    "shap":         shap.__version__,
    "matplotlib":   matplotlib.__version__,
    "seaborn":      sns.__version__,
    "openpyxl":     openpyxl.__version__,
}

version_df = pd.DataFrame(
    list(packages.items()),
    columns=["Package", "Version"]
)

print(version_df.to_string(index=False))

---
## 0.3 — Load the Dataset

Load `cs-training.csv` from `data/raw/`. The dataset includes an unnamed first column (row index from the original Kaggle export) — we'll load it and drop it.

In [ ]:
import os

# Path to the raw training data
RAW_DATA_PATH = os.path.join("..", "data", "raw", "cs-training.csv")

# Confirm the file exists before loading
assert os.path.exists(RAW_DATA_PATH), (
    f"Dataset not found at {RAW_DATA_PATH}. "
    f"Please download cs-training.csv from Kaggle and place it in data/raw/."
)

df = pd.read_csv(RAW_DATA_PATH)

# Drop the unnamed index column if present (Kaggle export artefact)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 0.4 — Column Names and Data Types

Verify that all 11 expected columns are present and confirm their data types. Any surprises here (e.g., a numeric column loaded as `object`) would indicate parsing issues.

In [ ]:
# Expected columns (after dropping the unnamed index)
EXPECTED_COLUMNS = [
    "SeriousDlqin2yrs",
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]

# Check all expected columns are present
missing_cols = set(EXPECTED_COLUMNS) - set(df.columns)
extra_cols = set(df.columns) - set(EXPECTED_COLUMNS)

if missing_cols:
    print(f"⚠️  MISSING columns: {missing_cols}")
elif extra_cols:
    print(f"⚠️  UNEXPECTED extra columns: {extra_cols}")
else:
    print(f"✅ All {len(EXPECTED_COLUMNS)} expected columns are present.")

print("\nData types:")
print(df.dtypes)

---
## 0.5 — Null Value Summary

Check which columns have missing values and how many. Based on the data documentation, we expect significant missingness in `MonthlyIncome` (~20%) and some in `NumberOfDependents` (~2.5%). These will be handled in Stage 2.

In [ ]:
null_summary = pd.DataFrame({
    "Null Count":   df.isnull().sum(),
    "Null %":       (df.isnull().sum() / len(df) * 100).round(2),
    "Non-Null":     df.notnull().sum(),
})

# Show only columns that have at least one null, plus a total row
null_summary_filtered = null_summary[null_summary["Null Count"] > 0]

if null_summary_filtered.empty:
    print("✅ No missing values in any column.")
else:
    print(f"Columns with missing values ({len(null_summary_filtered)}):")
    print(null_summary_filtered)
    print(f"\nTotal rows: {len(df):,}")

---
## 0.6 — Target Variable Distribution

The target `SeriousDlqin2yrs` is binary: 1 = the borrower experienced serious delinquency (90+ days past due) within 2 years. We expect roughly 6.7% of borrowers to be in the positive (default) class.

This class imbalance is important — it means we **cannot use accuracy** as a meaningful metric. A model that always predicts "no default" would be ~93% accurate but completely useless. We'll use AUC-ROC, KS statistic, and Gini coefficient instead.

In [ ]:
target_counts = df["SeriousDlqin2yrs"].value_counts()
default_rate = df["SeriousDlqin2yrs"].mean()

print("Target distribution:")
print(f"  0 (No default):   {target_counts[0]:>7,} borrowers  ({(1 - default_rate)*100:.1f}%)")
print(f"  1 (Defaulted):    {target_counts[1]:>7,} borrowers  ({default_rate*100:.1f}%)")
print(f"\n  Default rate:     {default_rate*100:.2f}%")
print(f"  Class ratio:      1 : {(1 - default_rate) / default_rate:.1f} (defaults : non-defaults)")

---
## 0.7 — Quick Sanity Check

Print the first few rows and basic descriptive statistics to confirm the data looks reasonable at a glance.

In [ ]:
df.head()

In [ ]:
df.describe().T

---
## ✅ Environment Check Complete

**Summary of checks:**
- All packages imported successfully
- Dataset loaded from `data/raw/cs-training.csv`
- Shape, columns, and data types confirmed
- Missing value pattern identified (MonthlyIncome and NumberOfDependents)
- Target distribution verified (~6.7% default rate)

**Next step:** Proceed to `01_eda.ipynb` for exploratory data analysis.